In [ ]:
# setup
! uv pip install -U fire omegaconf wandb jax[tpu] -q
! git clone --depth=1 https://github.com/martin-marek/picodo.git
! python /content/picodo/download_fineweb.py 'fineweb' 26 # 2.6B tokens

In [ ]:
# imports
%cd /content/picodo
from omegaconf import OmegaConf
from train import DEFAULT_CFG, train_and_evaluate


In [ ]:
# training (124M)
c = OmegaConf.merge(OmegaConf.create(DEFAULT_CFG), {'ds_path': '~/datasets/fineweb_gpt2.bin', 'model': {'D': 768, 'L': 12, 'T': 1024, 'V': 50257}})
c.model.activ_dtype = 'bfloat16' # reduced precision increases arithmetic intensity
c.model.H = 256 # TPU v6e has 256x256 MXU
c.opt.batch_size = 8 # smallest batch size maximizing MFU
c.opt.peak_lr = 0.002
c.opt.b2 = 0.999 # small batch size requires high b2 (arxiv.org/abs/2507.07101)
c.opt.weight_decay = 0.02 # small batch size requires small wd
c.wandb_mode = 'disabled' # ['disabled', 'offline', 'online']
train_and_evaluate(c)